In [2]:
import serial
from pymodbus.client import ModbusSerialClient
from MFC import *


def initCOM(port='COM3',
        baudrate=19200,
        bytesize=8,
        parity='N',
        stopbits=1,
        timeout=0.1):
    ser = serial.Serial(
        port=port,  # 实际的串口设备端口
        baudrate=baudrate,
        bytesize=bytesize,
        parity=parity,
        stopbits=stopbits,
        timeout=timeout
    )
    return ser

def main():   
    # 设置串口参数
    ser = initCOM()
    
    addr = 0x20
    data = getInstantFlowRateSend(addr)
    data = addCRC(data)
    data = mergeData2Bytes(data)
    #data.tobytes()
    print(data)
    success_bytes = ser.write(data) 
    # b表示bytes类型, 直接发送字符串报错
    print(success_bytes) # 发送数据长度
    
    res = ser.read(7)
    reslist = list(res)
    listTestByteAsHex = [int(hex(x).split('x')[-1]) for x in reslist]
    print(listTestByteAsHex)


In [3]:
ser = initCOM()

In [5]:
import time
from MFC import*

rd = setValveFlowRate(ser, 32, 0.75)
print(rd)

time.sleep(3)

addr, rf = getInstantFlowRate(ser,32)
print(rf)

[32, 6, 0, 2, 160, 0, 86, 187]
0.75


In [4]:
addr, rf = getInstantFlowRate(ser,32)
print(rf)

addr, rf = getInstantFlowRate(ser,33)
print(rf)

0.009857177734375
-3.0517578125e-05


In [24]:
rd = setValveFlowRate(ser, 32, 0.63)
print(rd)

time.sleep(3)

addr, rf = getInstantFlowRate(ser,32)
print(rf)

[32, 6, 0, 2, 144, 163, 2, 194]
0.629852294921875


In [ ]:
import sys
import os

# Get the absolute path of the current script's directory
current_dir = os.path.dirname(os.path.abspath(os.getcwd()))

sys.path.insert(0, current_dir)

from PID import *



Parent directory: c:\实验室
Current directory: c:\实验室\ThermalDataAnalysis


In [ ]:
from MFC import*

addr = 0x20
data = getInstantFlowRateSend(addr)
data = addCRC(data)
data = mergeData2Bytes(data)
#data.tobytes()
print(data)
success_bytes = ser.write(data) 
# b表示bytes类型, 直接发送字符串报错
print(success_bytes) # 发送数据长度

res = ser.read(7)
reslist = list(res)
print(res)
listTestByteAsHex = [hex(x).split('x')[-1] for x in reslist]
print(listTestByteAsHex)

print("check CRC:{}".format(checkCRC(reslist)))

print(getInstantFlowRateParse(reslist))

In [ ]:
addr = [0x23]
funcCode = [0x06]
paraAddr = [0x00,0x00]
para = [0x00,0x24]


data = [item for sublist in [addr,funcCode,paraAddr, para] for item in sublist]


crc_result = getCRC(data)
print(data)
print('%#x'%crc_result)

In [ ]:
addr = [0x23]
funcCode = [0x03]
paraAddr = [0x00,0x01]
para = [0x00,0x01]


data = [item for sublist in [addr,funcCode,paraAddr, para] for item in sublist]


crc_result = getCRC(data)
print(data)
print('%#x'%crc_result)

In [ ]:
addr = [0x24]
funcCode = [0x06]
paraAddr = [0x00,0x00]
para = [0x00,0x23]


data = [item for sublist in [addr,funcCode,paraAddr, para] for item in sublist]


crc_result = getCRC(data)
print(data)
print('%#x'%crc_result)

In [ ]:
addr = [0x20]
funcCode = [0x03]
paraAddr = [0x00,0x03]
para = [0x00,0x01]


data = [item for sublist in [addr,funcCode,paraAddr, para] for item in sublist]


crc_result = getCRC(data)
print(data)
print('%#x'%crc_result)

In [ ]:
def mergeData(data):
    dataLength = len(data)
    message = 0
    for i in range(dataLength):
        message = (message << 8) + data[i]
    return message

def setBaudRate(addr, baudRate):
    baudRates = [1200,2400,4800,9600,19200]
    assert baudRate in baudRates
    if baudRate == 1200:
        para = 0x04B0
    elif baudRate == 2400:
        para = 0x0960
    elif baudRate == 4800:
        para = 0x12C0
    elif baudRate == 9600:
        para = 0x2580
    elif baudRate == 19200:
        para = 0x4B00
    data = [addr, 0x06,0x00,0x01,para&0xff00, para&0x00ff]
    return data

def setAddr(addr, newAddr):
    assert newAddr >= 32
    assert newAddr <= 94
    data = [addr, 0x06,0x00,0x00,0x00, newAddr]
    return data

def setValveValue(addr, value):
    assert value >= 0
    assert value <= 1
    hexValue = 0x4000 + int(value * (0xC000 - 0x4000))
    data = [addr, 0x06,0x00,0x02,hexValue&0xff00, hexValue&0x00ff]

    return data

def setValveStatus(addr, status):
    assert status in [0,1,2]
    data = [addr, 0x06,0x00,0x05,0x00, status]
    crc = getCRC(data)
    data.append(crc&0xff00)
    data.append(crc&0x00ff)
    return data
    

In [ ]:
data = setAddr(0x20,0x21)
crc = getCRC(data)
print('%#x'%crc)
data.append(crc&0x00ff)
data.append((crc&0xff00)>>8)
print(data)
print('%#x'%mergeData(data))

In [ ]:
a = b' \x03\x00\x03\x00\x01r\xbb'
bl = list(a)
b = [hex(x) for x in bl]
print(bl)
print(b)